# Streaming

Structured output allows agents to return data in a specific, predictable format. Instead of parsing natural language responses, you get <b>structured data in the form of JSON objects, Pydantic models, or dataclasses</b> that your application can directly use.

LangChain’s create_agent handles structured output automatically. The user sets their desired structured output schema, and when the model generates the structured data, it’s captured, validated, and returned in the 'structured_response' key of the agent’s state.

In [ ]:
def create_agent(
    ...
    response_format: Union[
        ToolStrategy[StructuredResponseT],
        ProviderStrategy[StructuredResponseT],
        type[StructuredResponseT],
    ]

# Response Format

Controls how the agent returns structured data:
- ToolStrategy[StructuredResponseT]: Uses tool calling for structured output
- ProviderStrategy[StructuredResponseT]: Uses provider-native structured output
- type[StructuredResponseT]: Schema type - automatically selects best strategy based on model capabilities
None: No structured output

When a schema type is provided directly, LangChain automatically chooses:
- <b>ProviderStrategy for models supporting native structured output</b> (e.g. OpenAI, Anthropic, or Grok). 

^^^^ Meaning that these model have structured_output argument in their native api already!!!!
- <b>ToolStrategy for all other models.</b>


The structured response is returned in the structured_response key of the agent’s final state.

## Provider strategy

Some model providers support structured output natively through their APIs (e.g. OpenAI, Grok, Gemini). This is the most reliable method when available.

To use this strategy, configure a ProviderStrategy:

In [ ]:
class ProviderStrategy(Generic[SchemaT]):
    schema: type[SchemaT]


schema: required

The schema defining the structured output format. Supports:

- Pydantic models: BaseModel subclasses with field validation
- Dataclasses: Python dataclasses with type annotations
- TypedDict: Typed dictionary classes
- JSON Schema: Dictionary with JSON schema specification

LangChain <b>automatically uses ProviderStrategy when you pass a schema type directly to create_agent.</b> response_format and the model supports native structured output:

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [ ]:
from langchain.agents import create_agent


contact_info_schema = {
    "type": "object",
    "description": "Contact information for a person.",
    "properties": {
        "name": {"type": "string", "description": "The name of the person"},
        "email": {"type": "string", "description": "The email address of the person"},
        "phone": {"type": "string", "description": "The phone number of the person"}
    },
    "required": ["name", "email", "phone"]
}

agent = create_agent(
    model="gpt-5",
    tools=tools,
    response_format=ProviderStrategy(contact_info_schema)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

Provider-native structured output provides high reliability and strict validation because the model provider enforces the schema. Use it when available.

If the provider natively supports structured output for your model choice, it is functionally equivalent to write response_format=ProductReview instead of response_format=ProviderStrategy(ProductReview). In either case, if structured output is not supported, the agent will fall back to a tool calling strategy.

## Tool calling strategy

For models that don’t support native structured output, LangChain uses tool calling to achieve the same result. This works with all models that support tool calling, which is most modern models.

To use this strategy, configure a ToolStrategy:

In [ ]:
class ToolStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    tool_message_content: str | None
    handle_errors: Union[
        bool,
        str,
        type[Exception],
        tuple[type[Exception], ...],
        Callable[[Exception], str],
    ]

# This is already in  >>> from langchain.agents.structured_output import ToolStrategy


Args

<b>schema</b>: required

The schema defining the structured output format. Supports:
- Pydantic models: BaseModel subclasses with field validation
- Dataclasses: Python dataclasses with type annotations
- TypedDict: Typed dictionary classes
- JSON Schema: Dictionary with JSON schema specification
- Union types: Multiple schema options. The model will choose the most appropriate schema based on the context.


<b>tool_message_content</b>

Custom content for the tool message returned when structured output is generated. If not provided, defaults to a message showing the structured response data.

<b>handle_errors</b>

Error handling strategy for structured output validation failures. Defaults to True. (because it's not supported by the native api itself, langchain use another strategy to make the return formatted as a structured)

- True: Catch all errors with default error template
- str: Catch all errors with this custom message
- type[Exception]: Only catch this exception type with default message
- tuple[type[Exception], ...]: Only catch these exception types with default message
- Callable[[Exception], str]: Custom function that returns error message
- False: No retry, let exceptions propagate

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductReview(BaseModel):
    """Analysis of a product review."""
    rating: int | None = Field(description="The rating of the product", ge=1, le=5)
    sentiment: Literal["positive", "negative"] = Field(description="The sentiment of the review")
    key_points: list[str] = Field(description="The key points of the review. Lowercase, 1-3 words each.")

agent = create_agent(
    model="gpt-5",
    tools=tools,
    response_format=ToolStrategy(ProductReview)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'"}]
})
result["structured_response"]
# ProductReview(rating=5, sentiment='positive', key_points=['fast shipping', 'expensive'])

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductReview(BaseModel):
    """Analysis of a product review."""
    rating: int | None = Field(description="The rating of the product", ge=1, le=5)
    sentiment: Literal["positive", "negative"] = Field(description="The sentiment of the review")
    key_points: list[str] = Field(description="The key points of the review. Lowercase, 1-3 words each.")

class CustomerComplaint(BaseModel):
    """A customer complaint about a product or service."""
    issue_type: Literal["product", "service", "shipping", "billing"] = Field(description="The type of issue")
    severity: Literal["low", "medium", "high"] = Field(description="The severity of the complaint")
    description: str = Field(description="Brief description of the complaint")

agent = create_agent(
    model="gpt-5",
    tools=tools,
    response_format=ToolStrategy(Union[ProductReview, CustomerComplaint])
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'"}]
})
result["structured_response"]
# ProductReview(rating=5, sentiment='positive', key_points=['fast shipping', 'expensive'])

## Custom tool message content (ToolStrategy)

The tool_message_content parameter allows you to customize the message that appears in the conversation history when structured output is generated:

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class MeetingAction(BaseModel):
    """Action items extracted from a meeting transcript."""
    task: str = Field(description="The specific task to be completed")
    assignee: str = Field(description="Person responsible for the task")
    priority: Literal["low", "medium", "high"] = Field(description="Priority level")

agent = create_agent(
    model="gpt-5",
    tools=[],
    response_format=ToolStrategy(
        schema=MeetingAction,
        tool_message_content="Action item captured and added to meeting notes!"
    )
)

agent.invoke({
    "messages": [{"role": "user", "content": "From our meeting: Sarah needs to update the project timeline as soon as possible"}]
})

In [ ]:
# ================================ Human Message =================================

# From our meeting: Sarah needs to update the project timeline as soon as possible
# ================================== Ai Message ==================================
# Tool Calls:
#   MeetingAction (call_1)
#  Call ID: call_1
#   Args:
#     task: Update the project timeline
#     assignee: Sarah
#     priority: high
# ================================= Tool Message =================================
# Name: MeetingAction

# Action item captured and added to meeting notes!

Without tool_message_content, our final ToolMessage would be:

In [ ]:
# ================================= Tool Message =================================
# Name: MeetingAction

# Returning structured response: {'task': 'update the project timeline', 'assignee': 'Sarah', 'priority': 'high'}

## Error handling (ToolStrategy)

Models can make mistakes when generating structured output via tool calling. LangChain provides intelligent retry mechanisms to handle these errors automatically.
​
### Multiple structured outputs error
When a model incorrectly calls multiple structured output tools, the agent provides error feedback in a ToolMessage and prompts the model to retry:

In [ ]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ContactInfo(BaseModel):
    name: str = Field(description="Person's name")
    email: str = Field(description="Email address")

class EventDetails(BaseModel):
    event_name: str = Field(description="Name of the event")
    date: str = Field(description="Event date")

agent = create_agent(
    model="gpt-5",
    tools=[],
    response_format=ToolStrategy(Union[ContactInfo, EventDetails])  # Default: handle_errors=True
)

agent.invoke({
    "messages": [{"role": "user", "content": "Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th"}]
})

In [ ]:
# ================================ Human Message =================================

# Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th
# None
# ================================== Ai Message ==================================
# Tool Calls:
#   ContactInfo (call_1)
#  Call ID: call_1
#   Args:
#     name: John Doe
#     email: john@email.com
#   EventDetails (call_2)
#  Call ID: call_2
#   Args:
#     event_name: Tech Conference
#     date: March 15th
# ================================= Tool Message =================================
# Name: ContactInfo

# Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.
#  Please fix your mistakes.
# ================================= Tool Message =================================
# Name: EventDetails

# Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.
#  Please fix your mistakes.
# ================================== Ai Message ==================================
# Tool Calls:
#   ContactInfo (call_3)
#  Call ID: call_3
#   Args:
#     name: John Doe
#     email: john@email.com
# ================================= Tool Message =================================
# Name: ContactInfo

# Returning structured response: {'name': 'John Doe', 'email': 'john@email.com'}

### Schema validation error 

(e.g. exceed possible value of the field look at rating below)

When structured output doesn’t match the expected schema, the agent provides specific error feedback:

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductRating(BaseModel):
    rating: int | None = Field(description="Rating from 1-5", ge=1, le=5) # possible value = 1-5
    comment: str = Field(description="Review comment")

agent = create_agent(
    model="gpt-5",
    tools=[],
    response_format=ToolStrategy(ProductRating),  # Default: handle_errors=True
    system_prompt="You are a helpful assistant that parses product reviews. Do not make any field or value up."
)

agent.invoke({
    "messages": [{"role": "user", "content": "Parse this: Amazing product, 10/10!"}] # < 10 is exceed 5
})

In [ ]:
# ================================ Human Message =================================

# Parse this: Amazing product, 10/10!
# ================================== Ai Message ==================================
# Tool Calls:
#   ProductRating (call_1)
#  Call ID: call_1
#   Args:
#     rating: 10
#     comment: Amazing product
# ================================= Tool Message =================================
# Name: ProductRating

# Error: Failed to parse structured output for tool 'ProductRating': 1 validation error for ProductRating.rating
#   Input should be less than or equal to 5 [type=less_than_equal, input_value=10, input_type=int].
#  Please fix your mistakes.
# ================================== Ai Message ==================================
# Tool Calls:
#   ProductRating (call_2)
#  Call ID: call_2
#   Args:
#     rating: 5
#     comment: Amazing product
# ================================= Tool Message =================================
# Name: ProductRating

# Returning structured response: {'rating': 5, 'comment': 'Amazing product'}

# Error handling strategies

You can customize how errors are handled using the <b>handle_errors</b> parameter:

Custom error message:

In [ ]:
ToolStrategy(
    schema=ProductRating,
    handle_errors="Please provide a valid rating between 1-5 and include a comment."
)

If handle_errors is a string, the agent will always prompt the model to re-try with a fixed tool message:

In [ ]:
# ================================= Tool Message =================================
# Name: ProductRating

# Please provide a valid rating between 1-5 and include a comment.

### Handle specific exceptions only:

In [ ]:
ToolStrategy(
    schema=ProductRating,
    handle_errors=ValueError  # Only retry on ValueError, raise others
)

If handle_errors is an <b>exception type, the agent will only retry (using the default error message) if the exception raised is the specified type. In all other cases, the exception will be raised.</b>

### Custom error handler function:

In [ ]:
from langchain.agents.structured_output import StructuredOutputValidationError
from langchain.agents.structured_output import MultipleStructuredOutputsError

def custom_error_handler(error: Exception) -> str:
    if isinstance(error, StructuredOutputValidationError):
        return "There was an issue with the format. Try again.
    elif isinstance(error, MultipleStructuredOutputsError):
        return "Multiple structured outputs were returned. Pick the most relevant one."
    else:
        return f"Error: {str(error)}"


agent = create_agent(
    model="gpt-5",
    tools=[],
    response_format=ToolStrategy(
                        schema=Union[ContactInfo, EventDetails],
                        handle_errors=custom_error_handler
                    )  # Default: handle_errors=True
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th"}]
})

for msg in result['messages']:
    # If message is actually a ToolMessage object (not a dict), check its class name
    if type(msg).__name__ == "ToolMessage":
        print(msg.content)
    # If message is a dictionary or you want a fallback
    elif isinstance(msg, dict) and msg.get('tool_call_id'):
        print(msg['content'])

On StructuredOutputValidationError:

In [ ]:
# ================================= Tool Message =================================
# Name: ToolStrategy

# There was an issue with the format. Try again.

On MultipleStructuredOutputsError:

In [ ]:
# ================================= Tool Message =================================
# Name: ToolStrategy

# Multiple structured outputs were returned. Pick the most relevant one.

On other errors:

In [ ]:
# ================================= Tool Message =================================
# Name: ToolStrategy

# Error: <error message>

### No error handling:

In [ ]:
response_format = ToolStrategy(
    schema=ProductRating,
    handle_errors=False  # All errors raised
)

How LangChain Handles Errors with a String
When a LangChain agent executes a tool and that tool raises an exception (an error), the agent needs to decide what to do next. If you set handle_errors="That tool failed. Try a different approach.", here is what happens:

1. Tool Execution Fails: The model chooses a tool (e.g., Google Search) and provides input.

2. Error Occurs: The tool execution throws an exception (e.g., the API key is wrong, or the input format was invalid).

3. The Observation/Tool Output is Overridden: Instead of the agent seeing the standard, often lengthy and cryptic, error message from the tool, the agent framework substitutes the tool's output/observation with a special message. This message is constructed as follows:

The actual error message passed back to the LLM will be:

'Tool XXX failed with error: [Your handle_errors string]'

- Example: If your string is "Please re-try your approach, as the previous tool call failed.", the LLM will receive an "Observation" like:

Observation: Tool google_search failed with error: Please re-try your approach, as the previous tool call failed.

4. Model Re-tries: The LLM, seeing this clean, non-technical instruction in the "Observation" from the previous turn's prompt, is now essentially being told:

- "The last tool failed."

- "Here is an instruction on how to proceed (re-try)."

This makes the LLM less likely to crash or give up and more likely to formulate a new Action in its next response, effectively attempting to fix the error and re-try the task.

<b>Key Takeaway</b>
- The agent doesn't change the initial instruction (the system prompt or user query).

- It changes the conversational history/turn by providing a clean, custom error message as the Observation after a failed tool call. This custom message is your handle_errors string, framed as a "fixed tool message" telling the model to continue.

so it's like returning the message we config like , failed tool call, please try another approach to the message history and feed back to the model itself ?

# What actually happen when toolstrategy is called (by Gemini)

Here is the step-by-step process of what actually happens when you invoke an agent with response_format=ToolStrategy(Schema):

The ToolStrategy Agent Workflow

The core process is managed by the agent runtime (often built on LangGraph), which follows a loop of reasoning and action.

1. Setup: Creating the "Fake" Tool

- Schema Conversion: LangChain takes your ProductReview Pydantic model and converts its definition (including field names, types, and descriptions) into a JSON Schema.

- Virtual Tool Creation: It then creates a single, internal, and non-executable function (a "tool") whose input arguments are defined by that JSON Schema. This virtual tool is usually named something generic or hidden, like __structured_output or respond.

- Tool List Injection: This virtual tool is secretly added to the list of available tools that the model is presented with for its final decision.

2. Model Call 1: The Decision to Finish

- When the agent has completed its main task (e.g., it has called all necessary external tools and gathered all the required information), it knows it's ready to generate the final answer.

- Prompting: LangChain sends the conversation history and the overall prompt to the LLM. Critically, the prompt is tailored to instruct the model: "When you have the final answer, you must call one of the available tools with the complete information."

3. Model Response 1: The "Fake" Tool Call
- The LLM, which is excellent at tool calling, generates a structured response that is not a free-text answer. Instead, it generates a JSON object requesting to call the virtual tool:

In [ ]:
{
  "tool_calls": [
    {
      "name": "__structured_output",
      "arguments": {
        "rating": 5,
        "sentiment": "positive",
        "key_points": ["fast processor", "long battery"]
      }
    }
  ]
}

4. Execution & Parsing (The Critical Step)

- Interception: LangChain's runtime intercepts this tool call.

- No Execution: It recognizes that __structured_output is the virtual tool for structured output and does not execute any real function.

- Parsing: It extracts the arguments dictionary (rating, sentiment, etc.).

- Validation: It attempts to parse and validate this dictionary against your original ProductReview Pydantic model.

5. Error Handling (The handle_errors feature)

- If the model's generated JSON is malformed (e.g., a comma is missing), the validation fails and raises an exception.

- The handle_errors=True setting in ToolStrategy catches this error.

- Retry: LangChain generates a Tool Message containing the error feedback and sends it back to the model: "Your last output failed validation. The error was: [Validation Error Message]. Please try again, ensuring your JSON is correct." This forces the model to self-correct and try again.

6. Final Output

- Once the validation is successful (either on the first try or after a retry), the parsed and validated ProductReview object is extracted from the loop and returned as the final structured_response of the agent.